# 05 — Legs & Formation Classification

## Goal
Every valid S&D zone has three parts: **Leg-In → Base → Leg-Out**.  
This notebook measures those legs and classifies each cluster into one of four formation types.

## What Is a "Leg"?
A leg is a **directional move** into or out of the base, measured by net price displacement.

$$\text{leg\_net} = \text{close}_{n} - \text{open}_{1}$$

where candles $1 \dots n$ are the consecutive non-base candles immediately before or after the cluster.

## Leg Strength Requirement
A leg must be **strong enough** to count:

$$\frac{|\text{leg\_net}|}{\text{avg\_ATR}} \geq 1.5$$

A weak leg means price was drifting, not impulsively moving — no zone is formed.

## Formation Types

Each zone is classified by the direction of Leg-In and Leg-Out:

| Formation | Leg-In Direction | Leg-Out Direction | Zone Type |
|-----------|-----------------|-------------------|-----------|
| **RBR** — Rally-Base-Rally | Up | Up | Demand |
| **DBD** — Drop-Base-Drop | Down | Down | Supply |
| **DBR** — Drop-Base-Rally | Down | Up | Demand |
| **RBD** — Rally-Base-Drop | Up | Down | Supply |

A "flat" or absent leg → no formation → no zone.

## Known Issues in this Notebook

### Lookahead Bug
When `base_start == 0`, computing `c[base_start - 1]` in Python gives `c[-1]` — the **last candle of the dataset** (future data).  
Fix: skip any cluster where `base_start == 0`.

### Scenario F (Pullback) Failure
Scenario F has a real leg-out, but a pullback candle occurs within the `DEPARTURE_CANDLES=3` measurement window.  
The net displacement shrinks to near zero → classified as `flat` → zone rejected.  
Fix (not yet implemented): use *maximum displacement* in the window instead of final-close displacement.

## Visualization
The chart shows clusters colored by formation type, with the leg-in and leg-out arrows annotated.


In [1]:
import pandas as pd
import numpy as np

LEG_CANDLES   = 3      # how many candles to look back/forward for a leg
LEG_ATR_MIN   = 1.5    # leg must move at least 1.5x ATR to count

df = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv", index_col=0, parse_dates=True)

o = df["open"].values
h = df["high"].values
l = df["low"].values
c = df["close"].values
atr = df["atr"].values

print(f"{len(df)} candles loaded")


98 candles loaded


In [2]:
FORMATION_MAP = {
    ("up",   "up"):   "RBR",   # Rally-Base-Rally  → demand zone
    ("down", "down"): "DBD",   # Drop-Base-Drop    → supply zone
    ("down", "up"):   "DBR",   # Drop-Base-Rally   → demand zone
    ("up",   "down"): "RBD",   # Rally-Base-Drop   → supply zone
}

def classify_move(net, avg_atr):
    threshold = LEG_ATR_MIN * avg_atr
    if net >= threshold:
        return "up"
    if net <= -threshold:
        return "down"
    return "flat"

def measure_legs(base_start, base_end):
    """
    Returns (leg_in_dir, leg_out_dir, leg_in_net, leg_out_net).
    leg-in  : open of the candle LEG_CANDLES before base  →  close of candle just before base
    leg-out : close of last base candle  →  close of candle LEG_CANDLES after base
    """
    avg_atr = atr[base_start:base_end + 1].mean()

    in_start = max(0, base_start - LEG_CANDLES)
    leg_in_net = c[base_start - 1] - o[in_start]   # ← index -1 bug lives here if base_start=0

    out_end = min(len(c) - 1, base_end + LEG_CANDLES)
    leg_out_net = c[out_end] - c[base_end]

    return (
        classify_move(leg_in_net,  avg_atr),
        classify_move(leg_out_net, avg_atr),
        round(leg_in_net,  3),
        round(leg_out_net, 3),
    )

print("Formation map:")
for (lin, lout), form in FORMATION_MAP.items():
    print(f"  leg-in={lin:5s}  leg-out={lout:5s}  →  {form}")


Formation map:
  leg-in=up     leg-out=up     →  RBR
  leg-in=down   leg-out=down   →  DBD
  leg-in=down   leg-out=up     →  DBR
  leg-in=up     leg-out=down   →  RBD


In [3]:
# Step through Scenario A manually so the numbers are visible
scenario = "A_demand_RBR"
rows = labeled[labeled["scenario"] == scenario]

base_rows  = rows[rows["note"].str.startswith("BASE")]
leg_in_row = rows[rows["note"].str.startswith("LEG-IN")].iloc[0]
leg_out_row = rows[rows["note"].str.startswith("LEG-OUT")].iloc[0]

base_start = df.index.get_loc(base_rows.index[0])
base_end   = df.index.get_loc(base_rows.index[-1])

lin_dir, lout_dir, lin_net, lout_net = measure_legs(base_start, base_end)
formation = FORMATION_MAP.get((lin_dir, lout_dir), "unknown")

avg_atr = atr[base_start:base_end + 1].mean()
threshold = LEG_ATR_MIN * avg_atr

print(f"Scenario A — base candles: {base_start} → {base_end}")
print(f"  avg ATR      : {avg_atr:.3f}")
print(f"  threshold    : {threshold:.3f}  (= {LEG_ATR_MIN}x ATR)")
print(f"  leg-in  net  : {lin_net:+.3f}  → {lin_dir}")
print(f"  leg-out net  : {lout_net:+.3f}  → {lout_dir}")
print(f"  formation    : {formation}")


Scenario A — base candles: 19 → 21
  avg ATR      : 1.587
  threshold    : 2.381  (= 1.5x ATR)
  leg-in  net  : +5.400  → up
  leg-out net  : +6.000  → up
  formation    : RBR


In [4]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"

# resolve legs + formation for every labeled scenario
ZONE_COLOR = {"demand": "rgba(38,166,154,0.18)", "supply": "rgba(239,83,80,0.18)"}
ZONE_BORDER = {"demand": "#26a69a", "supply": "#ef5350"}

def zone_type(formation):
    return "demand" if formation in ("RBR", "DBR") else "supply"

scenarios_out = {}
for scen in labeled["scenario"].unique():
    if scen == "warmup":
        continue
    rows = labeled[labeled["scenario"] == scen]
    base_rows = rows[rows["note"].str.startswith("BASE")]
    if base_rows.empty:
        continue
    bs = df.index.get_loc(base_rows.index[0])
    be = df.index.get_loc(base_rows.index[-1])
    if bs == 0:   # guard against the lookahead bug
        continue
    lin_dir, lout_dir, lin_net, lout_net = measure_legs(bs, be)
    form = FORMATION_MAP.get((lin_dir, lout_dir), "unknown")
    scenarios_out[scen] = dict(
        bs=bs, be=be,
        base_high=df["high"].iloc[bs:be+1].max(),
        base_low=df["low"].iloc[bs:be+1].min(),
        lin_dir=lin_dir, lout_dir=lout_dir,
        formation=form,
    )

# ── chart ──────────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
))

for scen, z in scenarios_out.items():
    if z["formation"] == "unknown":
        continue
    zt = zone_type(z["formation"])
    x0 = df.index[z["bs"]]
    x1 = df.index[z["be"]]
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor=ZONE_COLOR[zt],
        line=dict(color=ZONE_BORDER[zt], width=1, dash="dot"),
        annotation_text=z["formation"],
        annotation_font=dict(size=9, color=ZONE_BORDER[zt]),
        annotation_position="top left",
    )

fig.update_layout(
    title="Legs & Formation — all scenarios",
    height=560,
    plot_bgcolor=BG, paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.04),
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)

fig.show()
